# # ERGAST
Ergast é uma API pública e gratuita que fornece dados históricos completos da Fórmula 1. É uma das fontes mais confiáveis e abrangentes para dados de F1, mantida pela comunidade desde 2008.

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import json
from datetime import datetime
import time

spark.sql("USE CATALOG f1_lakehouse")
def fetch_ergast_data(season, endpoint):
    """Busca dados da Ergast API com flatten completo de estruturas aninhadas"""
    url = f"https://api.jolpi.ca/ergast/f1/{season}/{endpoint}.json"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        if "MRData" in data:
            if endpoint == "drivers" and "DriverTable" in data["MRData"]:
                items = data["MRData"]["DriverTable"].get("Drivers", [])

            elif endpoint == "constructors" and "ConstructorTable" in data["MRData"]:
                items = data["MRData"]["ConstructorTable"].get("Constructors", [])

            elif endpoint == "races" and "RaceTable" in data["MRData"]:
                items = data["MRData"]["RaceTable"].get("Races", [])
                # Flatten Circuit
                for race in items:
                    if "Circuit" in race and isinstance(race["Circuit"], dict):
                        race["circuitId"] = race["Circuit"].get("circuitId")
                        del race["Circuit"]

            elif endpoint == "results" and "RaceTable" in data["MRData"]:
                items = []
                races = data["MRData"]["RaceTable"].get("Races", [])

                for race in races:
                    race_results = race.get("Results", [])

                    for result in race_results:
                        # Flatten Driver
                        if "Driver" in result and isinstance(result["Driver"], dict):
                            result["driverId"] = result["Driver"].get("driverId")
                            del result["Driver"]

                        # Flatten Constructor
                        if "Constructor" in result and isinstance(result["Constructor"], dict):
                            result["constructorId"] = result["Constructor"].get("constructorId")
                            del result["Constructor"]

                        # Flatten Status
                        if "Status" in result and isinstance(result["Status"], dict):
                            result["statusId"] = result["Status"].get("statusId")
                            del result["Status"]

                        # Flatten Time (pode ser um objeto com value)
                        if "Time" in result and isinstance(result["Time"], dict):
                            result["time"] = result["Time"].get("time")
                            result["milliseconds"] = result["Time"].get("millis")
                            del result["Time"]

                        # Flatten FastestLap (IMPORTANTE - esse é o seu erro)
                        if "FastestLap" in result and isinstance(result["FastestLap"], dict):
                            fastest_lap = result["FastestLap"]
                            result["fastestLap"] = fastest_lap.get("rank")

                            # Flatten Time dentro de FastestLap
                            if "Time" in fastest_lap and isinstance(fastest_lap["Time"], dict):
                                result["fastestLapTime"] = fastest_lap["Time"].get("time")
                            else:
                                result["fastestLapTime"] = fastest_lap.get("time")

                            result["fastestLapSpeed"] = fastest_lap.get("AverageSpeed", {}).get("speed") if isinstance(fastest_lap.get("AverageSpeed"), dict) else fastest_lap.get("AverageSpeed")
                            del result["FastestLap"]

                        result["raceId"] = race.get("raceId", race.get("round"))
                        result["_season"] = season
                        items.append(result)
            else:
                items = []

            # Adiciona metadados
            for item in items:
                item["_ingested_at"] = datetime.now().isoformat()
                item["_source"] = "ergast_api"
                if "_season" not in item:
                    item["_season"] = season

            return items
        return []

    except Exception as e:
        print(f"Erro ao buscar {endpoint}/{season}: {e}")
        import traceback
        traceback.print_exc()
        return []


@dlt.table(
    name="bronze.drivers_raw",
    comment="Dados de pilotos da Ergast API",
    table_properties={"quality": "bronze"}
)
def drivers_raw():
    """Pipeline DLT para ingestão de dados de pilotos"""
    all_drivers = []
    seasons = list(range(1950, 2025))

    for season in seasons[:10]:
        drivers = fetch_ergast_data(season, "drivers")
        all_drivers.extend(drivers)
        time.sleep(0.5)

    if all_drivers:
        return spark.createDataFrame(all_drivers)
    else:
        return spark.createDataFrame(
            [],
            schema=StructType([
                StructField("driverId", StringType(), True),
                StructField("driverRef", StringType(), True),
                StructField("number", StringType(), True),
                StructField("code", StringType(), True),
                StructField("forename", StringType(), True),
                StructField("surname", StringType(), True),
                StructField("nationality", StringType(), True),
                StructField("dateOfBirth", StringType(), True),
                StructField("url", StringType(), True),
                StructField("_ingested_at", StringType(), True),
                StructField("_source", StringType(), True),
                StructField("_season", IntegerType(), True)
            ])
        )


@dlt.table(
    name="bronze.constructors_raw",
    comment="Dados de construtores da Ergast API",
    table_properties={"quality": "bronze"}
)
def constructors_raw():
    """Pipeline DLT para ingestão de dados de construtores"""
    all_constructors = []
    seasons = list(range(2023, 2025))

    for season in seasons:
        constructors = fetch_ergast_data(season, "constructors")
        all_constructors.extend(constructors)
        time.sleep(0.5)

    if all_constructors:
        return spark.createDataFrame(all_constructors)
    else:
        return spark.createDataFrame(
            [],
            schema=StructType([
                StructField("constructorId", StringType(), True),
                StructField("constructorRef", StringType(), True),
                StructField("name", StringType(), True),
                StructField("nationality", StringType(), True),
                StructField("url", StringType(), True),
                StructField("_ingested_at", StringType(), True),
                StructField("_source", StringType(), True),
                StructField("_season", IntegerType(), True)
            ])
        )


@dlt.table(
    name="bronze.races_raw",
    comment="Dados de corridas da Ergast API",
    table_properties={"quality": "bronze"}
)
def races_raw():
    """Pipeline DLT para ingestão de dados de corridas"""
    all_races = []
    seasons = list(range(2023, 2025))

    for season in seasons:
        races = fetch_ergast_data(season, "races")
        all_races.extend(races)
        time.sleep(0.5)

    schema = StructType([
        StructField("season", StringType(), True),
        StructField("round", StringType(), True),
        StructField("url", StringType(), True),
        StructField("raceName", StringType(), True),
        StructField("date", StringType(), True),
        StructField("time", StringType(), True),

        # =========================
        # SESSÕES (STRUCTS)
        # =========================
        StructField("FirstPractice", StructType([
            StructField("date", StringType(), True),
            StructField("time", StringType(), True),
        ]), True),

        StructField("SecondPractice", StructType([
            StructField("date", StringType(), True),
            StructField("time", StringType(), True),
        ]), True),

        StructField("ThirdPractice", StructType([
            StructField("date", StringType(), True),
            StructField("time", StringType(), True),
        ]), True),

        StructField("Qualifying", StructType([
            StructField("date", StringType(), True),
            StructField("time", StringType(), True),
        ]), True),

        StructField("Sprint", StructType([
            StructField("date", StringType(), True),
            StructField("time", StringType(), True),
        ]), True),

        StructField("SprintShootout", StructType([
            StructField("date", StringType(), True),
            StructField("time", StringType(), True),
        ]), True),

        # =========================
        # CAMPOS SIMPLES
        # =========================
        StructField("circuitId", StringType(), True),

        # =========================
        # METADADOS
        # =========================
        StructField("_ingested_at", StringType(), True),
        StructField("_source", StringType(), True),
        StructField("_season", IntegerType(), True),
    ])

    if all_races:
        return spark.createDataFrame(all_races, schema=schema)
    else:
        return spark.createDataFrame([], schema=schema)


@dlt.table(
    name="bronze.results_raw",
    comment="Dados de resultados da Ergast API",
    table_properties={"quality": "bronze"}
)
def results_raw():
    """Pipeline DLT para ingestão de dados de resultados"""
    all_results = []
    seasons = list(range(2023, 2025))

    for season in seasons:
        results = fetch_ergast_data(season, "results")
        all_results.extend(results)
        time.sleep(1)

    if all_results:
        return spark.createDataFrame(all_results)
    else:
        return spark.createDataFrame(
            [],
            schema=StructType([
                StructField("resultId", StringType(), True),
                StructField("raceId", StringType(), True),
                StructField("driverId", StringType(), True),
                StructField("constructorId", StringType(), True),
                StructField("number", StringType(), True),
                StructField("grid", StringType(), True),
                StructField("position", StringType(), True),
                StructField("positionText", StringType(), True),
                StructField("positionOrder", StringType(), True),
                StructField("points", StringType(), True),
                StructField("laps", StringType(), True),
                StructField("time", StringType(), True),
                StructField("milliseconds", StringType(), True),
                StructField("fastestLap", StringType(), True),
                StructField("rank", StringType(), True),
                StructField("fastestLapTime", StringType(), True),
                StructField("fastestLapSpeed", StringType(), True),
                StructField("statusId", StringType(), True),
                StructField("_ingested_at", StringType(), True),
                StructField("_source", StringType(), True),
                StructField("_season", IntegerType(), True)
            ])
        )